# LLM interaction alignment e2e with local Ollama

This notebook mirrors `inspect_popularity_alignment_e2e.ipynb`, but replaces the popularity scorer with `LLMInteractionYesNoScorer` and a local OpenAI-compatible Ollama client.

The task can be built on up to `MAX_USERS = 10_000` random eligible users. Local LLM scoring is separately capped by `MAX_LLM_CANDIDATE_GROUPS`, because a full 10k-user yes/no run means thousands of sequential model calls on `llama3.1:8b`.

In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import pandas as pd


def _add_repo_root_to_path() -> None:
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "pyproject.toml").exists() and (path / "runners").exists():
            for import_path in (str(path), str(path / "src")):
                if import_path not in sys.path:
                    sys.path.insert(0, import_path)
            return
    raise RuntimeError("Could not find beyond-click-sim repo root")


_add_repo_root_to_path()

from runners.in_distribution.interaction_prediction.task_builders import (
    data_root,
    repo_root,
)

REPO_ROOT = repo_root()
DATA_ROOT = data_root()

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
print(f"repo_root={REPO_ROOT}")
print(f"data_root={DATA_ROOT}")


repo_root=/Users/a.agafonov/Research/agent_simulator/repos/beyond-click-sim
data_root=/Users/a.agafonov/Research/agent_simulator/data/canonical


In [2]:
from beyond_click_sim.data.canonical import CanonicalDataset
from beyond_click_sim.evaluation import binary_classification_metrics
from beyond_click_sim.llm_clients import ollama_client
from beyond_click_sim.scorers import LLMInteractionYesNoScorer
from beyond_click_sim.tasks import (
    AlignmentInteractionTaskBuilder,
    DatasetFilter,
    NonInteractionCandidateSampler,
    RandomFractionSplitter,
    split_xy,
)

In [3]:
# MovieLens is Agent4Rec-like; Steam checks the same path on a larger dataset.
RUN_DATASETS = ("ml-1m", "steam")

VERSION = "v1"
SEED = 0
MIN_INTERACTIONS = 10
MAX_USERS = 10_000

NEGATIVE_RATIO = 1
TRAIN_FRACTION = 0.7
VAL_FRACTION = 0.1
TEST_FRACTION = 0.2

# Set to None for a full run over one candidate group per selected user.
# Keep this small for the first local llama3.1:8b smoke/e2e run.
MAX_LLM_CANDIDATE_GROUPS = 25
MAX_LLM_ERRORS = 3

OLLAMA_MODEL = "llama3.1:8b"
MAX_HISTORY_ITEMS = 20
TEMPERATURE = 0.0
MAX_TOKENS = 256

In [4]:
DATASET_PROMPT_COLUMNS = {
    "ml-1m": {
        "history_context_columns": ("rating",),
        "history_description_columns": ("item_title", "item_genres", "rating"),
        "candidate_description_columns": ("item_title", "item_genres"),
    },
    "steam": {
        "history_context_columns": ("playtime_forever",),
        "history_description_columns": (
            "item_title",
            "item_genres_json",
            "item_tags_json",
            "playtime_forever",
        ),
        "candidate_description_columns": (
            "item_title",
            "item_genres_json",
            "item_tags_json",
        ),
    },
}


class MinInteractionsRandomUserFilter(DatasetFilter):
    """Keep users with enough interactions, then sample a reproducible user subset."""

    def __init__(
        self,
        *,
        min_interactions: int,
        max_users: int | None,
        seed: int = 0,
        user_column: str = "user_id",
    ) -> None:
        self.min_interactions = min_interactions
        self.max_users = max_users
        self.seed = seed
        self.user_column = user_column
        self.eligible_users = 0
        self.selected_users = 0

    def filter(
        self,
        users: pd.DataFrame,
        items: pd.DataFrame,
        interactions: pd.DataFrame,
    ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        counts = interactions.groupby(self.user_column, sort=False).size()
        eligible = counts[counts >= self.min_interactions].index.to_series(index=None)
        self.eligible_users = int(len(eligible))

        if self.max_users is not None and len(eligible) > self.max_users:
            selected = eligible.sample(n=self.max_users, random_state=self.seed)
        else:
            selected = eligible.sample(frac=1.0, random_state=self.seed)

        selected_users = set(selected.tolist())
        self.selected_users = int(len(selected_users))
        return (
            users[users[self.user_column].isin(selected_users)].copy(),
            items.copy(),
            interactions[interactions[self.user_column].isin(selected_users)].copy(),
        )


def load_canonical_dataset(dataset_name: str) -> CanonicalDataset:
    root = DATA_ROOT / dataset_name / VERSION
    return CanonicalDataset(
        name=dataset_name,
        version=VERSION,
        root=root,
        users_path=root / "users.parquet",
        items_path=root / "items.parquet",
        interactions_path=root / "interactions.parquet",
        manifest_path=root / "manifest.json",
    )


def build_alignment_task(dataset: CanonicalDataset):
    columns = DATASET_PROMPT_COLUMNS[dataset.name]
    builder = AlignmentInteractionTaskBuilder(
        name=f"{dataset.name}-interaction-alignment-random-ollama-m{NEGATIVE_RATIO}",
        dataset_filter=MinInteractionsRandomUserFilter(
            min_interactions=MIN_INTERACTIONS,
            max_users=MAX_USERS,
            seed=SEED,
        ),
        splitter=RandomFractionSplitter(
            train_fraction=TRAIN_FRACTION,
            val_fraction=VAL_FRACTION,
            test_fraction=TEST_FRACTION,
            seed=SEED,
            group_column="user_id",
        ),
        sampler=NonInteractionCandidateSampler(
            negative_ratio=NEGATIVE_RATIO,
            seed=SEED,
        ),
        history_context_columns=columns["history_context_columns"],
    )
    return builder.build(dataset)


def candidate_summary(frame: pd.DataFrame, task) -> dict[str, int | float]:
    group_column = task.schema.candidate_group_column
    sampled_column = task.schema.sampled_column
    target_column = task.schema.target_column
    if group_column is None or sampled_column is None:
        raise ValueError("candidate and sampled columns are required")
    group_sizes = frame.groupby(group_column).size()
    positives = frame.groupby(group_column)[target_column].sum()
    sampled = frame.groupby(group_column)[sampled_column].sum()
    return {
        "candidate_sets": int(len(group_sizes)),
        "rows_min": int(group_sizes.min()),
        "rows_max": int(group_sizes.max()),
        "positives_min": float(positives.min()),
        "positives_max": float(positives.max()),
        "sampled_min": int(sampled.min()),
        "sampled_max": int(sampled.max()),
    }


def select_one_candidate_group_per_user(
    frame: pd.DataFrame,
    *,
    user_column: str,
    candidate_group_column: str,
    max_groups: int | None,
    seed: int,
) -> pd.DataFrame:
    groups = frame[[candidate_group_column, user_column]].drop_duplicates()
    groups = groups.sample(frac=1.0, random_state=seed).drop_duplicates(user_column)
    if max_groups is not None:
        groups = groups.head(max_groups)
    selected_groups = set(groups[candidate_group_column].tolist())
    return frame[frame[candidate_group_column].isin(selected_groups)].copy()


def score_candidate_groups(
    scorer: LLMInteractionYesNoScorer,
    X: pd.DataFrame,
    *,
    candidate_group_column: str,
    max_errors: int,
) -> tuple[pd.Series, list[dict[str, object]], float]:
    groups = list(X.groupby(candidate_group_column, sort=False))
    scores = pd.Series(index=X.index, dtype=float, name="score")
    errors: list[dict[str, object]] = []
    started = time.perf_counter()

    for position, (group_id, group) in enumerate(groups, start=1):
        try:
            group_scores = scorer.score(group)
            scores.loc[group_scores.index] = group_scores
        except Exception as error:
            errors.append({"candidate_group": group_id, "error": repr(error)})
            if len(errors) >= max_errors:
                print(f"  stopping after {len(errors)} LLM errors")
                break

        if position == 1 or position % 10 == 0 or position == len(groups):
            elapsed = time.perf_counter() - started
            print(
                f"  scored {position}/{len(groups)} groups in {elapsed:.1f}s; "
                f"errors={len(errors)}"
            )

    return scores, errors, time.perf_counter() - started


def run_llm_alignment(dataset_name: str) -> tuple[dict[str, object], dict[str, object]]:
    started = time.perf_counter()
    dataset = load_canonical_dataset(dataset_name)
    task = build_alignment_task(dataset)
    build_seconds = time.perf_counter() - started

    target_column = task.schema.target_column
    group_column = task.schema.candidate_group_column
    user_column = task.schema.id_columns[0]
    if group_column is None:
        raise ValueError("task must define candidate_group_column")

    X_train, y_train = split_xy(task.train, target_column=target_column)
    X_test, y_test = split_xy(task.test, target_column=target_column)
    X_eval = select_one_candidate_group_per_user(
        X_test,
        user_column=user_column,
        candidate_group_column=group_column,
        max_groups=MAX_LLM_CANDIDATE_GROUPS,
        seed=SEED,
    )
    y_eval = y_test.loc[X_eval.index]

    columns = DATASET_PROMPT_COLUMNS[dataset_name]
    scorer = LLMInteractionYesNoScorer(
        client=ollama_client(),
        model=OLLAMA_MODEL,
        history_description_columns=columns["history_description_columns"],
        candidate_description_columns=columns["candidate_description_columns"],
        user_column=user_column,
        candidate_group_column=group_column,
        max_history_items=MAX_HISTORY_ITEMS,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    scorer.fit(X_train, y_train)

    scores, errors, score_seconds = score_candidate_groups(
        scorer,
        X_eval,
        candidate_group_column=group_column,
        max_errors=MAX_LLM_ERRORS,
    )

    valid_scores = scores.dropna()
    valid_y = y_eval.loc[valid_scores.index]
    if len(valid_scores) > 0:
        metrics = binary_classification_metrics(valid_y, valid_scores.astype(int))
    else:
        metrics = {
            "accuracy": None,
            "precision": None,
            "recall": None,
            "f1": None,
            "n": 0,
            "n_positive": 0,
            "n_predicted_positive": 0,
        }

    scored_rows = pd.concat(
        [
            X_eval.loc[:, [user_column, "item_id", group_column, *columns["candidate_description_columns"]]],
            y_eval.rename("target"),
            scores,
        ],
        axis=1,
    )

    row = {
        "dataset": dataset_name,
        "model": OLLAMA_MODEL,
        "negative_ratio": NEGATIVE_RATIO,
        "candidate_ratio": f"1:{NEGATIVE_RATIO}",
        "max_users_requested": MAX_USERS,
        "users_after_filter": int(task.manifest["users"]),
        "items": int(task.manifest["items"]),
        "train_rows": int(len(task.train)),
        "test_rows": int(len(task.test)),
        "test_candidate_sets": int(task.test[group_column].nunique()),
        "llm_candidate_sets_selected": int(X_eval[group_column].nunique()),
        "llm_rows_selected": int(len(X_eval)),
        "llm_rows_scored": int(valid_scores.notna().sum()),
        "llm_errors": int(len(errors)),
        "build_seconds": round(build_seconds, 3),
        "score_seconds": round(score_seconds, 3),
        **{f"test_{key}": value for key, value in metrics.items()},
    }
    artifacts = {
        "task": task,
        "X_train": X_train,
        "y_train": y_train,
        "X_eval": X_eval,
        "y_eval": y_eval,
        "scores": scores,
        "errors": errors,
        "scored_rows": scored_rows,
        "test_candidate_summary": candidate_summary(task.test, task),
    }
    return row, artifacts

In [5]:
rows = []
artifacts_by_dataset = {}

for dataset_name in RUN_DATASETS:
    print(f"Running {dataset_name} with {OLLAMA_MODEL}...")
    row, artifacts = run_llm_alignment(dataset_name)
    rows.append(row)
    artifacts_by_dataset[dataset_name] = artifacts
    print(
        f"  done: users={row['users_after_filter']}, "
        f"selected_groups={row['llm_candidate_sets_selected']}, "
        f"scored_rows={row['llm_rows_scored']}, "
        f"errors={row['llm_errors']}, "
        f"f1={row['test_f1']}"
    )

results = pd.DataFrame(rows)
results

Running ml-1m with llama3.1:8b...


  scored 1/25 groups in 4.6s; errors=0


  scored 10/25 groups in 30.9s; errors=0


  scored 20/25 groups in 60.7s; errors=0


  scored 25/25 groups in 74.6s; errors=0


...

  scored 10/25 groups in 87.5s; errors=0


  scored 20/25 groups in 176.1s; errors=0


  scored 25/25 groups in 218.5s; errors=0


  done: users=10000, selected_groups=25, scored_rows=50, errors=0, f1=0.12903225806451613


,dataset,model,negative_ratio,candidate_ratio,max_users_requested,users_after_filter,items,train_rows,test_rows,test_candidate_sets,llm_candidate_sets_selected,llm_rows_selected,llm_rows_scored,llm_errors,build_seconds,score_seconds,test_accuracy,test_precision,test_recall,test_f1,test_n,test_n_positive,test_n_predicted_positive
0,ml-1m,llama3.1:8b,1,1:1,10000,6040,3883,694335,404902,202451,25,50,50,0,19.740,74.646,0.44,0.285714,0.08,0.125000,50,25,7
1,steam,llama3.1:8b,1,1:1,10000,10000,10978,605453,359454,179727,25,50,50,0,20.075,218.502,0.46,0.333333,0.08,0.129032,50,25,6


In [6]:
metric_columns = [
    "dataset",
    "model",
    "negative_ratio",
    "users_after_filter",
    "items",
    "train_rows",
    "test_rows",
    "test_candidate_sets",
    "llm_candidate_sets_selected",
    "llm_rows_selected",
    "llm_rows_scored",
    "llm_errors",
    "build_seconds",
    "score_seconds",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_n",
    "test_n_positive",
    "test_n_predicted_positive",
]
results.loc[:, metric_columns]

,dataset,model,negative_ratio,users_after_filter,items,train_rows,test_rows,test_candidate_sets,llm_candidate_sets_selected,llm_rows_selected,llm_rows_scored,llm_errors,build_seconds,score_seconds,test_accuracy,test_precision,test_recall,test_f1,test_n,test_n_positive,test_n_predicted_positive
0,ml-1m,llama3.1:8b,1,6040,3883,694335,404902,202451,25,50,50,0,19.740,74.646,0.44,0.285714,0.08,0.125000,50,25,7
1,steam,llama3.1:8b,1,10000,10978,605453,359454,179727,25,50,50,0,20.075,218.502,0.46,0.333333,0.08,0.129032,50,25,6


In [7]:
dataset_name = RUN_DATASETS[0]
artifacts = artifacts_by_dataset[dataset_name]

print("test candidate summary:")
display(pd.DataFrame([artifacts["test_candidate_summary"]]))

print("scored rows sample:")
display(artifacts["scored_rows"].head(20))

if artifacts["errors"]:
    print("LLM errors:")
    display(pd.DataFrame(artifacts["errors"]))

test candidate summary:


,candidate_sets,rows_min,rows_max,positives_min,positives_max,sampled_min,sampled_max
0,202451,2,2,1.0,1.0,1,1


scored rows sample:


,user_id,item_id,candidate_group,item_title,item_genres,target,score
39840,1486,3617,candidate:1486:ml-1m:row:0000245999,Road Trip (2000),Comedy,1,0.0
39841,1486,885,candidate:1486:ml-1m:row:0000245999,Bogus (1996),Children's|Drama|Fantasy,0,0.0
41384,1501,2329,candidate:1501:ml-1m:row:0000248868,American History X (1998),Drama,1,0.0
41385,1501,1692,candidate:1501:ml-1m:row:0000248868,Alien Escape (1995),Horror|Sci-Fi,0,0.0
60464,1737,62,candidate:1737:ml-1m:row:0000291748,Mr. Holland's Opus (1995),Drama,1,0.0
60465,1737,3014,candidate:1737:ml-1m:row:0000291748,Bustin' Loose (1981),Comedy,0,1.0
75576,1926,2025,candidate:1926:ml-1m:row:0000325658,Lolita (1997),Drama|Romance,1,0.0
75577,1926,1866,candidate:1926:ml-1m:row:0000325658,"Big Hit, The (1998)",Action|Comedy,0,0.0
82222,2005,1093,candidate:2005:ml-1m:row:0000340018,"Doors, The (1991)",Drama|Musical,1,0.0
82223,2005,1120,candidate:2005:ml-1m:row:0000340018,"People vs. Larry Flynt, The (1996)",Drama,0,0.0
